In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:  # aqui mantemos menor que o limite
            return True
        return False
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [4]:
analize = Analizer(0.8)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
1,model_22_7_6,0.799486,-0.836105,0.340663,0.598058,0.637291,0.278628,2.551404,0.081214,1.519816,0.800515,1.005176,0.527853,1.052883,0.550325,232.555752,372.726472,"Hidden Size=[8, 9], regularizer=0.2, learning_..."
2,model_22_7_7,0.799363,-0.847556,0.228061,0.594452,0.631059,0.278800,2.567315,0.095084,1.533454,0.814269,1.230776,0.528015,1.052915,0.550494,232.554523,372.725243,"Hidden Size=[8, 9], regularizer=0.2, learning_..."
4,model_22_7_5,0.798941,-0.824160,0.450572,0.601830,0.643589,0.279387,2.534805,0.067676,1.505554,0.786615,0.746223,0.528571,1.053027,0.551073,232.550316,372.721036,"Hidden Size=[8, 9], regularizer=0.2, learning_..."
6,model_29_7_14,0.798650,-0.984789,0.881157,0.580498,0.850252,0.279791,2.758010,0.104013,0.137038,0.120525,18.943125,0.528953,1.230115,0.551471,92.547425,147.396837,"Hidden Size=[11], regularizer=0.2, learning_ra..."
7,model_29_7_15,0.798581,-0.989923,0.872886,0.508211,0.831085,0.279886,2.765144,0.111251,0.160652,0.135952,18.800588,0.529043,1.230193,0.551565,92.546744,147.396156,"Hidden Size=[11], regularizer=0.2, learning_ra..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
552,model_33_7_2,0.753183,0.047015,0.538684,0.662929,0.574085,0.361418,1.395469,1.118476,0.279872,0.699174,3.485974,0.601180,1.348448,0.626774,84.035441,134.009350,"Hidden Size=[10], regularizer=0.2, learning_ra..."
553,model_1_4_2,0.753143,-0.138991,0.910513,0.810158,0.876196,0.361476,1.667841,0.184611,0.204403,0.194507,1.688699,0.601229,1.987427,0.626824,62.035119,98.601394,"Hidden Size=[3, 4], regularizer=0.2, learning_..."
554,model_31_3_9,0.753082,-0.094188,0.771386,0.601738,0.678982,0.361566,1.602235,0.449477,1.018757,0.734116,1.845151,0.601303,1.042633,0.626902,328.034624,526.711383,"Hidden Size=[10, 11], regularizer=0.05, learni..."
558,model_15_6_20,0.752599,-0.410527,0.571363,0.349374,0.761687,0.343782,1.960032,0.658536,0.512048,0.585292,14.792312,0.586329,1.116424,0.611290,152.135497,243.551184,"Hidden Size=[6, 7], regularizer=0.05, learning..."
